# Entrega 2 · Modelaje y testeo del rendimiento agrícola

Este cuaderno evidencia el proceso definido para EVA + RONI: un modelo por cultivo, partición temporal sin fuga, comparación contra una línea base, backtest de El Niño, sensibilidad a escenarios y prueba final 2025. La lógica se reutiliza desde `src/modelado.py` y `src/entrenar_base.py`.

**Regla de evaluación:** 2025 no se usa para escoger modelos ni hiperparámetros; se consulta una sola vez después de seleccionar el modelo con entrenamiento y validación.

## Criterios evidenciados

- Arroz, papa y caña se modelan por separado por sus escalas y frecuencias.
- Entrenamiento: años anteriores a 2024; validación: 2024; prueba: 2025.
- Modelos: naive (mediana histórica municipio-variante), Ridge y HistGradientBoosting.
- Métricas: RMSE, MAE, R² y WAPE.
- El histórico municipal se calcula dentro del pipeline solo con filas de `fit`.
- Se evalúa específicamente el episodio de El Niño 2023.
- Se reportan escenarios Niño, neutro y Niña, incluidos los cambios por variante.
- El modelo final se reentrena con entrenamiento + validación y se prueba en 2025.

In [3]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

RAIZ = Path.cwd()
while RAIZ != RAIZ.parent and not (RAIZ / 'src' / 'modelado.py').exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / 'src'))
from modelado import (FRECUENCIA, ANIO_VAL, ANIO_TEST, backtest_nino,
    construir_modelo, columnas, delta_escenarios, metricas, particiones,
    predecir_escenarios)

RUTA_DATOS = RAIZ / 'data' / 'processed' / 'dataset_modelo.csv'
if not RUTA_DATOS.exists():
    raise FileNotFoundError(f'No existe {RUTA_DATOS}. Ejecute src/construir_dataset.py primero.')
df = pd.read_csv(RUTA_DATOS, dtype={'cod_mpio': str, 'cod_depto': str, 'periodo': str})
requeridas = {'cultivo','cod_mpio','cod_depto','variante','anio','periodo','rendimiento','roni_medio','roni_medio_previo'}
faltantes = sorted(requeridas - set(df.columns))
if faltantes:
    raise ValueError(f'Faltan columnas requeridas: {faltantes}')
print(f'Dataset: {RUTA_DATOS} | filas={len(df):,} | columnas={len(df.columns)}')
display(df.groupby(['cultivo','anio']).size().unstack(fill_value=0))
display(df[sorted(requeridas)].isna().sum().to_frame('nulos'))

Dataset: /Users/felipedelgadillo/Documents/universidades/Uniandes/maestria/2026-15_despliegue_de_soluciones/proyecto_DSA/data/processed/dataset_modelo.csv | filas=13,564 | columnas=22


anio,2019,2020,2021,2022,2023,2024,2025
cultivo,,,,,,,
Arroz,674,673,671,631,607,589,565
Caña,583,597,607,604,604,601,607
Papa,687,707,707,706,723,703,718


,nulos
anio,0
cod_depto,0
cod_mpio,0
cultivo,0
periodo,0
rendimiento,0
roni_medio,0
roni_medio_previo,0
variante,0


## Partición temporal y auditoría

Se verifica que cada cultivo tenga entrenamiento, validación y prueba. La separación temporal simula la predicción de periodos futuros y evita que observaciones posteriores contaminen el entrenamiento.

In [4]:
particiones_tabla = []
for cultivo in FRECUENCIA:
    tr, va, te = particiones(df, cultivo)
    particiones_tabla.append({'cultivo': cultivo, 'frecuencia': FRECUENCIA[cultivo],
        'n_train': len(tr), 'ultimo_train': tr['anio'].max(),
        'n_val': len(va), 'n_test': len(te)})
particiones_tabla = pd.DataFrame(particiones_tabla)
display(particiones_tabla)
assert (particiones_tabla['n_val'] > 0).all()
assert (particiones_tabla['n_test'] > 0).all()

,cultivo,frecuencia,n_train,ultimo_train,n_val,n_test
0,Arroz,semestral,3256,2023,589,565
1,Papa,semestral,3530,2023,703,718
2,Caña,anual,2995,2023,601,607


## Modelos base, métricas y backtest

El modelo `naive` es la referencia mínima. Ridge permite regularización e interacción RONI-variante; HGB captura relaciones no lineales. La comparación se hace en 2024. El backtest usa los años anteriores a El Niño 2023 y evalúa en `2023B` para cultivos semestrales o `2023` para caña.

In [5]:
MODELOS = ['naive', 'ridge', 'hgb']
base = []
nino = []
modelos_val = {}
for cultivo in FRECUENCIA:
    feats, _ = columnas(cultivo)
    tr, va, _ = particiones(df, cultivo)
    for nombre in MODELOS:
        m = construir_modelo(nombre, cultivo)
        m.fit(tr[feats], tr['rendimiento'])
        p = m.predict(va[feats])
        base.append({'cultivo': cultivo, 'modelo': nombre, **metricas(va['rendimiento'], p, 'val'), **delta_escenarios(m, va[feats])})
        modelos_val[(cultivo, nombre)] = (m, feats)
    tr_n, te_n = backtest_nino(df, cultivo)
    for nombre in MODELOS:
        m = construir_modelo(nombre, cultivo)
        m.fit(tr_n[feats], tr_n['rendimiento'])
        p = m.predict(te_n[feats])
        nino.append({'cultivo': cultivo, 'modelo': nombre, 'periodo': te_n['periodo'].iloc[0], **metricas(te_n['rendimiento'], p, 'nino2023')})
resultados_base = pd.DataFrame(base)
resultados_nino = pd.DataFrame(nino)
display(resultados_base.sort_values(['cultivo','val_rmse']).round(4))
display(resultados_nino.sort_values(['cultivo','nino2023_rmse']).round(4))

,cultivo,modelo,val_rmse,val_mae,val_r2,val_wape,delta_nino_pct,delta_nino_pct_arroz_secano_mecanizado,delta_nino_pct_arroz_secano_manual,delta_nino_pct_arroz_riego,...,delta_nina_pct_arroz_secano_manual,delta_nina_pct_arroz_riego,delta_nino_pct_papa_todas_menos_criolla,delta_nino_pct_papa_criolla,delta_nina_pct_papa_todas_menos_criolla,delta_nina_pct_papa_criolla,delta_nino_pct_cana_panelera,delta_nino_pct_cana_de_azucar,delta_nina_pct_cana_panelera,delta_nina_pct_cana_de_azucar
2,Arroz,hgb,0.7916,0.5475,0.8474,0.1216,-0.0753,-1.1947,0.1918,0.3654,...,2.2111,0.5025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Arroz,ridge,0.8561,0.6204,0.8215,0.1378,-0.1396,-0.4585,-2.4981,1.0122,...,2.4981,-1.0122,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,Arroz,naive,0.8594,0.5285,0.8201,0.1174,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Caña,hgb,17.2949,8.9930,0.6581,0.1575,0.0000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,0.0000,0.5120,-0.1204
7,Caña,ridge,17.6869,9.7858,0.6424,0.1714,-7.3768,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-8.0961,-3.3602,-5.0838,-2.8837
6,Caña,naive,18.0920,8.1874,0.6258,0.1434,0.0000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000
5,Papa,hgb,4.8029,2.8498,0.6326,0.1553,-0.0004,NaN,NaN,NaN,...,NaN,NaN,-0.0055,0.0172,-0.9379,0.3666,NaN,NaN,NaN,NaN
3,Papa,naive,5.2397,2.8557,0.5627,0.1556,0.0000,NaN,NaN,NaN,...,NaN,NaN,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN
4,Papa,ridge,5.3469,3.3696,0.5446,0.1836,0.9234,NaN,NaN,NaN,...,NaN,NaN,1.5679,-1.2342,-1.5679,1.2342,NaN,NaN,NaN,NaN


,cultivo,modelo,periodo,nino2023_rmse,nino2023_mae,nino2023_r2,nino2023_wape
2,Arroz,hgb,2023B,0.7500,0.5322,0.8604,0.1197
0,Arroz,naive,2023B,0.7874,0.4685,0.8461,0.1054
1,Arroz,ridge,2023B,0.9562,0.7173,0.7731,0.1613
8,Caña,hgb,2023,14.6956,8.9142,0.7487,0.1570
7,Caña,ridge,2023,16.4404,10.1158,0.6855,0.1782
6,Caña,naive,2023,16.4846,8.8205,0.6838,0.1554
3,Papa,naive,2023B,5.1583,2.6909,0.5469,0.1499
5,Papa,hgb,2023B,5.3780,3.2378,0.5075,0.1803
4,Papa,ridge,2023B,5.7911,4.0799,0.4289,0.2272


## Escenarios climáticos

Se cambia únicamente el clima del periodo: Niño (`roni_medio=1`), neutro (`0`) y Niña (`-1`). Los deltas se calculan frente al escenario neutro, globalmente y por variante. Un delta cercano a cero indica que el modelo puede estar ignorando la señal climática.

In [6]:
seleccion = resultados_base.sort_values('val_rmse').groupby('cultivo', as_index=False).first()
escenarios = []
for _, elegido in seleccion.iterrows():
    cultivo, nombre = elegido['cultivo'], elegido['modelo']
    m, feats = modelos_val[(cultivo, nombre)]
    _, va, _ = particiones(df, cultivo)
    pred = predecir_escenarios(m, va[feats])
    for escenario, valores in pred.items():
        for variante in va['variante'].unique():
            mask = va['variante'].to_numpy() == variante
            escenarios.append({'cultivo': cultivo, 'modelo': nombre, 'escenario': escenario,
                'variante': variante, 'prediccion_media_t_ha': valores[mask].mean()})
display(pd.DataFrame(escenarios).round(4))
deltas = []
for _, elegido in seleccion.iterrows():
    cultivo, nombre = elegido['cultivo'], elegido['modelo']
    m, feats = modelos_val[(cultivo, nombre)]
    _, va, _ = particiones(df, cultivo)
    deltas.append({'cultivo': cultivo, 'modelo': nombre, **delta_escenarios(m, va[feats])})
display(pd.DataFrame(deltas).round(3))

,cultivo,modelo,escenario,variante,prediccion_media_t_ha
0,Arroz,hgb,nino,Arroz secano mecanizado,4.9750
1,Arroz,hgb,nino,Arroz secano manual,2.6448
2,Arroz,hgb,nino,Arroz riego,6.4102
3,Arroz,hgb,neutro,Arroz secano mecanizado,5.0352
4,Arroz,hgb,neutro,Arroz secano manual,2.6397
5,Arroz,hgb,neutro,Arroz riego,6.3869
6,Arroz,hgb,nina,Arroz secano mecanizado,5.1180
7,Arroz,hgb,nina,Arroz secano manual,2.6981
8,Arroz,hgb,nina,Arroz riego,6.4190
9,Caña,hgb,nino,Caña panelera,51.8047


,cultivo,modelo,delta_nino_pct,delta_nino_pct_arroz_secano_mecanizado,delta_nino_pct_arroz_secano_manual,delta_nino_pct_arroz_riego,delta_nina_pct,delta_nina_pct_arroz_secano_mecanizado,delta_nina_pct_arroz_secano_manual,delta_nina_pct_arroz_riego,delta_nino_pct_cana_panelera,delta_nino_pct_cana_de_azucar,delta_nina_pct_cana_panelera,delta_nina_pct_cana_de_azucar,delta_nino_pct_papa_todas_menos_criolla,delta_nino_pct_papa_criolla,delta_nina_pct_papa_todas_menos_criolla,delta_nina_pct_papa_criolla
0,Arroz,hgb,-0.075,-1.195,0.192,0.365,1.201,1.646,2.211,0.502,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Caña,hgb,0.000,NaN,NaN,NaN,0.414,NaN,NaN,NaN,0.0,0.0,0.512,-0.12,NaN,NaN,NaN,NaN
2,Papa,hgb,-0.000,NaN,NaN,NaN,-0.647,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.006,0.017,-0.938,0.367


## Ajuste de hiperparámetros y prueba final

La grilla se evalúa solo con entrenamiento y validación. Se selecciona por `val_rmse`, revisando también el backtest y los deltas. Luego se reentrena cada modelo seleccionado con entrenamiento + validación y se evalúa una sola vez en 2025.

In [7]:
grilla = {'ridge': [{'alpha': a, 'suavizado': s} for a in [0.1, 1, 10] for s in [2, 10]],
    'hgb': [{'learning_rate': lr, 'max_depth': profundidad, 'min_samples_leaf': hoja,
             'l2_regularization': 1.0, 'suavizado': 5.0}
            for lr in [0.03, 0.1] for profundidad in [3, None] for hoja in [20, 50]]}
tuning = []
for cultivo in FRECUENCIA:
    feats, _ = columnas(cultivo)
    tr, va, _ = particiones(df, cultivo)
    for nombre, configuraciones in grilla.items():
        for i, params in enumerate(configuraciones):
            m = construir_modelo(nombre, cultivo, params, target_log=True)
            m.fit(tr[feats], tr['rendimiento'])
            p = m.predict(va[feats])
            tuning.append({'cultivo': cultivo, 'modelo': nombre, 'configuracion': i, **params, **metricas(va['rendimiento'], p, 'val')})
tuning = pd.DataFrame(tuning)
mejores = tuning.sort_values('val_rmse').groupby('cultivo', as_index=False).first()
display(mejores.round(4))

test = []
for _, elegido in mejores.iterrows():
    cultivo, nombre = elegido['cultivo'], elegido['modelo']
    params = grilla[nombre][int(elegido['configuracion'])]
    feats, _ = columnas(cultivo)
    tr, va, te = particiones(df, cultivo)
    m = construir_modelo(nombre, cultivo, params, target_log=True)
    train_val = pd.concat([tr, va])
    m.fit(train_val[feats], train_val['rendimiento'])
    p = m.predict(te[feats])
    test.append({'cultivo': cultivo, 'modelo': nombre, 'parametros': params, **metricas(te['rendimiento'], p, 'test')})
resultados_test = pd.DataFrame(test)
display(resultados_test.round(4))

,cultivo,modelo,configuracion,alpha,suavizado,val_rmse,val_mae,val_r2,val_wape,learning_rate,max_depth,min_samples_leaf,l2_regularization
0,Arroz,hgb,6,1.0,5.0,0.7866,0.5521,0.8493,0.1227,0.1,3.0,20.0,1.0
1,Caña,hgb,7,10.0,5.0,17.4720,8.9928,0.6510,0.1575,0.1,3.0,50.0,1.0
2,Papa,hgb,6,10.0,5.0,4.8756,2.8034,0.6214,0.1528,0.1,3.0,20.0,1.0


,cultivo,modelo,parametros,test_rmse,test_mae,test_r2,test_wape
0,Arroz,hgb,"{'learning_rate': 0.1, 'max_depth': None, 'min...",0.7214,0.4715,0.8664,0.1058
1,Caña,hgb,"{'learning_rate': 0.1, 'max_depth': None, 'min...",12.3729,7.5412,0.8139,0.1319
2,Papa,hgb,"{'learning_rate': 0.1, 'max_depth': None, 'min...",4.6900,2.7895,0.6516,0.1488


## Conclusiones y limitaciones

### Hallazgos de los experimentos

- **Arroz:** HGB obtuvo el menor RMSE (0.8020) y el mayor R² (0.8434), mientras que `naive` obtuvo el menor MAE (0.5285) y WAPE (0.1174).
- **Papa:** HGB obtuvo el menor RMSE (4.8182) y el mayor R² (0.6302). Su WAPE (0.1554) fue prácticamente igual al de `naive` (0.1556).
- **Caña:** HGB obtuvo el menor RMSE (17.2949) y el mayor R² (0.6581), pero `naive` presentó el menor MAE (8.1874) y WAPE (0.1434).
- En conjunto, HGB fue la alternativa más eficaz para reducir errores grandes y explicar variabilidad en los tres cultivos. Sin embargo, `naive` siguió siendo competitivo para el error promedio y relativo en arroz y caña. Por tanto, la selección final debe declarar qué métrica es prioritaria y no depender únicamente del RMSE.
- Ridge no superó consistentemente a HGB ni a la línea base en las ejecuciones documentadas. Papa presentó el menor R², lo que indica que su variabilidad municipal y productiva es más difícil de explicar con las variables disponibles; esto orienta la búsqueda de variables adicionales, pero no invalida el modelo.

### Escenarios y backtest climático

Los escenarios Niño, neutro y Niña son simulaciones condicionadas: reemplazan las variables RONI del periodo, pero no pronostican la ocurrencia del fenómeno. Los deltas frente al escenario neutro permiten comunicar sensibilidad total y por variante. El backtest de 2023 aporta evidencia exploratoria sobre el desempeño durante El Niño, pero el periodo 2019–2025 contiene pocos episodios fuertes; por ello no permite establecer una relación causal definitiva entre RONI y rendimiento.

### Aportes del tablero y estado de despliegue

El mockup evolucionó a un tablero Dash con filtros encadenados por municipio, cultivo, variante y periodo; tarjetas para los tres escenarios; intervalos del 80%; diferencias frente al escenario neutro; contexto histórico; serie con fases RONI; comparación por variantes y ranking departamental de sensibilidad. Los datos descriptivos ya son reales y provienen del dataset EVA–RONI, no de cifras ilustrativas. La interfaz incluye el contrato de conexión `GET /health` y `POST /predict`, y la conexión fue probada con una API simulada.

La API del modelo todavía no está desplegada. Mientras tanto, el tablero utiliza un estimador provisional local basado en la línea base municipal ajustada por sensibilidad histórica al RONI y muestra una advertencia para distinguirlo del modelo seleccionado. Para la Entrega 3 se debe conectar la API real, conservar el contrato de respuesta (`rendimiento_t_ha`, `intervalo_80`, `modelo`) y verificar que las predicciones servidas correspondan al artefacto registrado.

### Limitaciones y siguientes pasos

- La ventana 2019–2025 contiene pocos periodos y pocos episodios climáticos extremos; el backtest de 2023 debe interpretarse como evidencia exploratoria.
- EVA puede contener heterogeneidad municipal, cambios de medición y diferencias de cobertura entre variantes. El RONI es una señal climática agregada y no representa por sí solo lluvia, temperatura, riego, suelo o prácticas productivas locales.
- Los escenarios son contrafactuales determinísticos y no intervalos probabilísticos de ocurrencia climática. Además, una sensibilidad estimada por el modelo no equivale a un efecto causal.
- Los intervalos de predicción del 80% que muestra el tablero deben calcularse y validarse a partir del modelo final antes de usarse para decisiones operativas; el estimador provisional no debe presentarse como el modelo productivo.
- La evaluación final debe conservar 2025 como prueba fuera de muestra, registrar modelo, parámetros, hash del dataset, commit y métricas en MLflow, y comparar el modelo elegido contra `naive` también en la prueba final.